# 05 — Data Pipelines, ETL/ELT, DAGs & Data Quality

**Day 2 | Data Engineering & AI Bootcamp**

Data pipelines are the infrastructure that keeps AI systems fed with clean, timely data.
This notebook covers: ETL vs ELT patterns, batch vs stream processing, DAG orchestration,
PySpark basics, and data quality validation with pandera / Great Expectations.

**PySpark requires Java 11+.** The demo gracefully skips if Java is unavailable.

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from day2.pipeline import (
    extract_from_source, etl_transform,
    elt_load_raw, elt_transform_in_warehouse,
    batch_processor, stream_processor,
    build_ingestion_dag, pyspark_demo, DAG, Task,
)
from day2.data_quality import (
    validate_completeness, validate_uniqueness, validate_distribution,
    run_validation_suite, great_expectations_concept,
    build_transaction_schema, validate_with_pandera,
)

sns.set_theme(style='whitegrid')
print('Setup complete ✓')

## 1. ETL vs ELT — The Architecture Decision

**ETL (Extract → Transform → Load):** Clean data *before* loading.
Traditional approach. Required when the target system can't handle raw/messy data.
Examples: legacy data warehouses, relational databases with strict constraints.

**ELT (Extract → Load → Transform):** Load raw data first, clean *inside* the warehouse.
Modern approach. Cloud warehouses (Databricks, Snowflake, BigQuery) are powerful enough to transform at scale.
Benefit: raw data is always preserved — re-run transforms without re-extracting.

> **Medallion Architecture (Databricks):**
> - **Bronze** = raw as-is from source (ELT load step)
> - **Silver** = cleaned, typed, deduplicated (ELT transform step)
> - **Gold**  = aggregated, business-ready (ELT analytical layer)

In [ ]:
# Extract raw records (simulates an API / database source)
records = extract_from_source(60)
print(f'Extracted {len(records)} raw records')
print(f'Sample: id={records[0].id}, amount={records[0].amount!r}, category={records[0].category}')
print(f'Notice: amount is a STRING like "1,234.56" or "N/A" — real source data is messy!')

print('\n--- ETL Pattern ---')
df_etl = etl_transform(records)  # transform before loading
print(f'ETL output: {len(df_etl)} rows, dtypes: amount={df_etl["amount"].dtype}')

print('\n--- ELT Pattern ---')
bronze = elt_load_raw(records)              # load raw
silver = elt_transform_in_warehouse(bronze) # transform in-warehouse
print(f'Bronze: {len(bronze)} rows | Silver: {len(silver)} rows')
print(f'Bronze has N/A values: {(bronze["amount"] == "N/A").sum()}')
print(f'Silver has N/A values: 0 (filtered in transform step)')

display(bronze.head(3))
display(silver.head(3))

In [ ]:
gold = batch_processor(silver)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bronze
axes[0].barh(['Rows'], [len(bronze)], color='#cd7f32', alpha=0.85)
axes[0].set_title('Bronze (raw)', fontweight='bold', fontsize=11)
axes[0].set_xlabel(f'{len(bronze)} rows — raw, unclean')

# Silver
axes[1].barh(['Rows'], [len(silver)], color='#C0C0C0', alpha=0.85)
axes[1].set_title('Silver (clean)', fontweight='bold', fontsize=11)
axes[1].set_xlabel(f'{len(silver)} rows — typed, validated')

# Gold
axes[2].barh(gold['category'][:5], gold['total_amount'][:5], color='#FFD700', alpha=0.85)
axes[2].set_title('Gold (aggregated)', fontweight='bold', fontsize=11)
axes[2].set_xlabel('Total Amount')

plt.suptitle('Medallion Architecture: Bronze → Silver → Gold', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Batch vs Stream Processing

**Batch processing:** process data in bulk on a schedule (hourly, daily, weekly).
High throughput, simple to implement, acceptable latency (minutes to hours).
Best for: model retraining, data warehouse loads, end-of-day reports.

**Stream processing:** process each record as it arrives. Low latency (milliseconds–seconds).
More complex, requires stateful computation (windowing, watermarks).
Best for: fraud detection (can't wait until end of day!), live dashboards, IoT alerts.

> **Unified APIs:** Apache Spark Structured Streaming uses the same DataFrame API for both.
> Switch from batch to stream by changing the source reader. Databricks DLT abstracts this further.

In [ ]:
print('--- Batch Processing ---')
t0 = time.perf_counter()
gold = batch_processor(silver)
elapsed = time.perf_counter() - t0

print(f'Processed {len(silver)} rows in {elapsed*1000:.1f}ms (all at once)')
print(f'Result: {len(gold)} aggregation rows')
display(gold.head(6))

In [ ]:
print('--- Stream Processing (sliding window simulation) ---')

# Simulate a real-time stream: records arrive one at a time
record_stream = (
    {'id': r.id, 'amount': float(r.amount.replace(',', ''))}
    for r in records if r.amount != 'N/A'
)

summaries = list(stream_processor(record_stream, window_size=5))
stream_df  = pd.DataFrame(summaries)
print(f'Emitted {len(summaries)} window summaries (one per 5 records)')
display(stream_df.head(5))

# Visualise rolling average
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(stream_df['total_processed'], stream_df['window_avg'],
        'b-o', markersize=5, label='Window Average')
ax.fill_between(stream_df['total_processed'],
                stream_df['window_min'], stream_df['window_max'],
                alpha=0.2, label='Window Range')
ax.set_xlabel('Total Records Processed')
ax.set_ylabel('Amount')
ax.set_title('Streaming: Rolling Window Average Over Time', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. DAG — Directed Acyclic Graph

A DAG represents a pipeline as a graph where **nodes are tasks** and **edges are dependencies**.
'Directed' means dependencies have a direction (A must finish before B).
'Acyclic' means no circular dependencies (prevents infinite loops).

Production DAG tools:
- **Apache Airflow:** Python-based, most popular, complex but powerful.
- **Databricks Workflows:** built-in job orchestration, drag-and-drop UI + YAML.
- **dbt:** DAG for SQL transformations, automatic dependency resolution.

> **Why DAGs?** They enable: parallel execution of independent tasks, retry on failure,
> dependency-aware scheduling, and visual debugging of pipeline failures.

In [ ]:
# Build and run the ingestion pipeline DAG
dag = build_ingestion_dag(records)

print('Tasks in DAG:')
for name, task in dag.tasks.items():
    deps = task.depends_on if task.depends_on else ['(start)']
    print(f'  {name:<20} → depends on: {deps}')

print()
outputs = dag.run()

print(f'\nFinal report: {outputs["generate_report"]}')

In [ ]:
# Build a custom RAG ingestion DAG
from day2.rag import SAMPLE_DOCUMENTS, chunk_corpus, Document

rag_dag = DAG('rag_ingestion')

rag_dag.add_task(Task(
    name       = 'load_docs',
    fn         = lambda _: SAMPLE_DOCUMENTS,
    depends_on = [],
    description = 'Load source documents',
))

rag_dag.add_task(Task(
    name       = 'chunk_docs',
    fn         = lambda docs: chunk_corpus(docs, chunk_size=50, overlap=10),
    depends_on = ['load_docs'],
    description = 'Chunk documents into segments',
))

rag_dag.add_task(Task(
    name       = 'count_chunks',
    fn         = lambda chunks: {'num_chunks': len(chunks), 'avg_words': sum(len(c.text.split()) for c in chunks) / len(chunks)},
    depends_on = ['chunk_docs'],
    description = 'Validate and count chunks',
))

rag_outputs = rag_dag.run()
print(f'RAG DAG result: {rag_outputs["count_chunks"]}')

## 4. PySpark Basics

PySpark is the Python API for Apache Spark — a distributed data processing framework.
Where Pandas runs on one machine with ~100M rows, Spark runs across a cluster with billions.
PySpark uses **lazy evaluation**: transformations build a plan, actions execute it.

Key concepts:
- **DataFrame:** distributed table, like Pandas but across multiple machines
- **Transformation:** lazy — `.filter()`, `.groupBy()`, `.join()` just build a plan
- **Action:** triggers execution — `.show()`, `.count()`, `.collect()`, `.write()`
- **Partition:** data is split into partitions, each processed on a different node

> **Requires Java 11+.** On Databricks, Spark is pre-installed — you just write `spark.read...`
> and Databricks handles the cluster. This demo shows the same API locally with 2 threads.

In [ ]:
spark_result = pyspark_demo(silver)

if isinstance(spark_result, dict):
    print(f'PySpark demo successful!')
    print(f'  Spark DataFrame rows: {spark_result["spark_rows"]}')
    print(f'  Schema: {spark_result["spark_schema"]}')
    print(f'  Parquet partitions written: {spark_result["parquet_parts"]}')
    print(f'\nGrouped result (category aggregation):')
    for row in spark_result.get('grouped_result', []):
        print(f'  {row}')
else:
    print(spark_result)
    print()
    print('PySpark code that would run with Java 11+:')
    print('''
  from pyspark.sql import SparkSession, functions as F
  spark = SparkSession.builder.appName("day2").master("local[*]").getOrCreate()

  # Convert pandas to Spark
  sdf = spark.createDataFrame(silver)

  # Lazy transformations (plan built, not executed)
  result = (
      sdf
      .filter(F.col("amount") > 100)
      .groupBy("category")
      .agg(F.count("*").alias("count"), F.avg("amount").alias("avg_amount"))
      .orderBy(F.desc("count"))
  )

  # Action — executes the full plan
  result.show()

  # Write as Parquet (production output format)
  sdf.write.parquet("/mnt/datalake/silver/transactions", mode="overwrite")
  ''')

## 5. Data Quality & Validation

Bad data causes bad models and wrong business decisions. The fix: validate data at every pipeline step.
**pandera** defines Python schemas — type constraints, value ranges, uniqueness, custom rules.
Run validation before processing: catch errors early, not after the model is deployed.

> **Great Expectations** is the enterprise version: generates HTML reports, tracks validation results
> over time, and integrates with Airflow/dbt. Use pandera in code, GE for stakeholder reporting.

In [ ]:
print('--- Validation on CLEAN data ---')
reports = run_validation_suite(silver)
for r in reports:
    print(r.summary())

In [ ]:
print('--- Validation on DATA WITH INTENTIONAL BUGS ---')
silver_bad = silver.copy()

# Bug 1: negative amount (data entry error)
silver_bad.loc[0, 'amount'] = -500.0

# Bug 2: invalid category (typo from source system)
silver_bad.loc[1, 'category'] = 'UNKNOWN_CATEGORY'

# Bug 3: duplicate transaction ID (double-processing bug)
silver_bad.loc[2, 'id'] = silver_bad.loc[0, 'id']

reports_bad = run_validation_suite(silver_bad)
for r in reports_bad:
    print(r.summary())
    print()

In [ ]:
# Distribution check
print('--- Distribution Check ---')
r_dist = validate_distribution(silver, 'amount', expected_min=0, expected_max=10000)
print(r_dist.summary())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of amount
axes[0].hist(silver['amount'], bins=20, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].axvline(silver['amount'].quantile(0.95), color='red', linestyle='--',
                label=f'P95 = {silver["amount"].quantile(0.95):.0f}')
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Count')
axes[0].set_title('Amount Distribution', fontweight='bold')
axes[0].legend()

# Category distribution
cat_counts = silver['category'].value_counts()
axes[1].bar(cat_counts.index, cat_counts.values, color=['#e74c3c','#3498db','#2ecc71','#9b59b6','#e67e22'],
            alpha=0.85, edgecolor='white')
axes[1].set_title('Category Distribution', fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 6. Great Expectations — Enterprise Data Quality

Great Expectations (GE) extends pandera-style validation with:
- **Expectation Suites:** named collections of assertions that live in version control
- **Data Docs:** auto-generated HTML reports showing pass/fail, shareable with stakeholders
- **Checkpoints:** scheduled runs that produce validation results tracked over time
- **Profiling:** auto-suggest expectations based on data statistics

> In Databricks, GE integrates with Delta Live Tables: every DLT pipeline can have
> expectations that halt or warn when data quality rules are violated.

In [ ]:
import pprint

ge_info = great_expectations_concept()

print('Great Expectations Core Concepts:')
for concept, desc in ge_info['core_concepts'].items():
    print(f'  {concept:<20}: {desc}')

print(f'\npandera vs Great Expectations:')
for k, v in ge_info['pandera_vs_ge'].items():
    print(f'  {k:<22}: {v}')

print(f'\nCommon Expectation types:')
for exp in ge_info['common_expectations']:
    print(f'  {exp}')

print(f'''
Great Expectations code (requires: pip install great-expectations):

  import great_expectations as ge

  context = ge.get_context()
  suite   = context.suites.add(ExpectationSuite(name="transactions_v1"))

  suite.add_expectation(ExpectationConfiguration(
      expectation_type = "expect_column_values_to_not_be_null",
      kwargs           = {{"column": "amount"}},
  ))
  suite.add_expectation(ExpectationConfiguration(
      expectation_type = "expect_column_values_to_be_between",
      kwargs           = {{"column": "amount", "min_value": 0, "max_value": 1_000_000}},
  ))

  # Run and generate HTML report
  results = context.run_checkpoint("transactions_checkpoint")
  context.open_data_docs()  # Opens browser with HTML report
''')